In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 512),
            nn.ReLU(),
        )

        self.policy_head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128,act_dim)
        )
        self.value_head = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self, x):
        x = self.shared(x)
        logits = self.policy_head(x)
        value = self.value_head(x)
        return logits, value

    def act(self, state):
        logits, value = self.forward(state)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        action = dist.sample()
        log_prob = dist.log_prob(action)

        return action, log_prob, value

    def evaluate(self, states, actions):
        logits, values = self.forward(states)
        probs = torch.softmax(logits, dim=-1)
        dist = torch.distributions.Categorical(probs)

        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()

        return log_probs, values.squeeze(-1), entropy

In [2]:
class RolloutBuffer:
    def __init__(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.dones = []
        self.values = []

    def clear(self):
        self.__init__()

    def compute_returns_advantages(self, gamma=0.99, lam=0.95, num_envs=4):
        T = len(self.rewards)
        rollout_steps = T // num_envs

        # --- reshape into (num_envs, rollout_steps) ---
        rewards = torch.tensor(self.rewards, dtype=torch.float32).view(num_envs, rollout_steps)
        dones = torch.tensor(self.dones, dtype=torch.float32).view(num_envs, rollout_steps)
        values = torch.tensor(self.values, dtype=torch.float32).view(num_envs, rollout_steps)

        returns = torch.zeros_like(rewards)
        advantages = torch.zeros_like(rewards)

        # --- compute GAE per environment ---
        for env in range(num_envs):
            gae = 0
            next_value = 0  

            for t in reversed(range(rollout_steps)):
                delta = (
                    rewards[env, t]
                    + gamma * next_value * (1 - dones[env, t])
                    - values[env, t]
                )

                gae = delta + gamma * lam * (1 - dones[env, t]) * gae

                advantages[env, t] = gae
                returns[env, t] = gae + values[env, t]

                next_value = values[env, t]

        # --- flatten back to original shape (T,) ---
        returns = returns.view(-1)
        advantages = advantages.view(-1)

        return returns, advantages

In [3]:
import soccer_twos
from gym_unity.envs import ActionFlattener


def make_env(worker_id,mode=None):
    if mode is None:
        env = soccer_twos.make(worker = worker_id,)
    if mode == "random":
        env = soccer_twos.make(
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    if mode == "still":
        env = soccer_twos.make(
            opponent_policy=lambda *_: 0, 
            variation=soccer_twos.EnvType.team_vs_policy,
            single_player=True,
            worker = worker_id,
            flatten_branched=True,
        )
    # print(env.action_space.nvec)
    return env

In [4]:
from torch.utils.tensorboard import SummaryWriter
import time

log_dir = f"runs/soccer_ppo_{int(time.time())}"
writer = SummaryWriter(log_dir=log_dir)
%load_ext tensorboard

In [10]:
%tensorboard --logdir runs/

Reusing TensorBoard on port 6006 (pid 921680), started 11:24:32 ago. (Use '!kill 921680' to kill it.)

In [6]:
import numpy as np

def info_reward(info, prev_ball_vel, team_sign):
    player_info = info["player_info"]
    ball_info = info["ball_info"]

    player_pos = player_info["position"]
    player_vel = player_info["velocity"]
    ball_pos = ball_info["position"]
    ball_vel = ball_info["velocity"]

    # =========================
    # --- helpers ---
    # =========================
    def normalize(v):
        return v / (np.linalg.norm(v) + 1e-8)

    def norm(v):
        return np.linalg.norm(v)

    # =========================
    # --- geometry ---
    # =========================
    to_ball = ball_pos - player_pos
    dist_ball = norm(to_ball)
    to_ball_dir = normalize(to_ball)

    ball_speed = norm(ball_vel)
    prev_speed = norm(prev_ball_vel)

    ball_dir = normalize(ball_vel)
    prev_dir = normalize(prev_ball_vel)

    goal_dir = np.array([team_sign, 0.0])
    own_goal_dir = np.array([-team_sign, 0.0])

    # =========================
    # 1. POSITIONING (behind ball)
    # =========================
    d_target = 1.5
    target_pos = ball_pos + own_goal_dir * d_target

    to_target = target_pos - player_pos
    dist_target = norm(to_target)

    r_position = np.exp(-dist_target) - 0.5   # [-0.5, 0.5]
    r_position *= 0.1

    # =========================
    # 2. MOVE TOWARD TARGET
    # =========================
    to_target_dir = normalize(to_target)
    r_move = 0.01 * np.dot(player_vel, to_target_dir)

    # =========================
    # 3. BALL → GOAL DIRECTION
    # =========================
    if ball_speed > 0.1:
        r_goal = 0.1 * np.dot(ball_dir, goal_dir)
    else:
        r_goal = 0.0

    # =========================
    # 4. DIRECTIONAL IMPACT (NEW)
    # =========================
    if ball_speed > 0.05 and prev_speed > 0.05:
        toward_goal_now = np.dot(ball_dir, goal_dir)
        toward_goal_prev = np.dot(prev_dir, goal_dir)

        if toward_goal_prev > 0:
            # was good → penalize reversing
            reverse = -np.dot(ball_dir, prev_dir)
            r_dir_impact = -0.01 * max(0.0, reverse)
        else:
            # was bad → reward correction
            r_dir_impact = 0.01 * toward_goal_now

        # scale by speed (important!)
        r_dir_impact *= np.clip(ball_speed / 2.0, 0, 1)
    else:
        r_dir_impact = 0.0

    # =========================
    # 5. IMPACT STRENGTH
    # =========================
    if 0.5 < dist_ball < 2.0:
        r_impact = 0.02 * ball_speed
    else:
        r_impact = 0.0

    # =========================
    # --- return components ---
    # =========================
    return r_position, r_move, r_goal, r_dir_impact, r_impact

In [ ]:
import torch.optim as optim
import numpy as np
# hyperparameters
NUM_ENVS = 1
ROLLOUT_STEPS = 512
EPOCHS = 10
BATCH_SIZE = 256
GAMMA = 0.99
LAMBDA = 0.95
CLIP_EPS = 0.2
LR = 3e-4


episode_rewards = []
current_rewards = [0 for _ in range(NUM_ENVS)]
global_step = 0


def collect_rollout(envs,model):
    buffer = RolloutBuffer()
    global global_step
    states = []
    for env in envs:
        states.append(env.reset()[:obs_dim])
    prev_ball_vels = [np.zeros(2) for _ in range(NUM_ENVS)]
    states = np.array(states)
    team_signs = [None for _ in range(NUM_ENVS)]

    for _ in range(ROLLOUT_STEPS):
        state_tensor = torch.tensor(states, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, values = model.act(state_tensor)

        next_states = []
        rewards = []
        dones = []

        for i, env  in enumerate(envs):
            action = actions[i].item()
            obs, reward, done, info = env.step(action)
            if team_signs[i] is None:
                player_x = info["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
                
            ball_vel = info["ball_info"]["velocity"]
            r_pos, r_move, r_goal, r_dir_impact, r_impact = info_reward(
                info,
                prev_ball_vels[i],
                team_signs[i]
            )
            prev_ball_vels[i] = ball_vel.copy()
            r = reward + r_pos + r_move + r_goal + r_dir_impact + r_impact
            writer.add_scalar("reward/env", reward, global_step)
            writer.add_scalar("reward/move", r_move, global_step)
            writer.add_scalar("reward/pos", r_pos, global_step)
            writer.add_scalar("reward/goal", r_goal, global_step)
            writer.add_scalar("reward/impact_dir", r_dir_impact, global_step)
            writer.add_scalar("reward/impact", r_impact, global_step)
            writer.add_scalar("reward/total", r, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs


            next_states.append(next_obs[:obs_dim])
            rewards.append(r)
            dones.append(done)

        buffer.states.extend(state_tensor)
        buffer.actions.extend(actions)
        buffer.log_probs.extend(log_probs)
        buffer.rewards.extend(rewards)
        buffer.dones.extend(dones)
        buffer.values.extend(values.detach().view(-1).tolist())

        states = np.array(next_states)
        global_step += NUM_ENVS

    return buffer


def ppo_update(buffer,optimizer):
    returns, advantages = buffer.compute_returns_advantages(GAMMA, LAMBDA)

    states = torch.stack(buffer.states)
    actions = torch.stack(buffer.actions)
    old_log_probs = torch.stack(buffer.log_probs)

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    for _ in range(EPOCHS):
        for i in range(0, len(states), BATCH_SIZE):
            s = states[i:i+BATCH_SIZE]
            a = actions[i:i+BATCH_SIZE]
            old_lp = old_log_probs[i:i+BATCH_SIZE]
            adv = advantages[i:i+BATCH_SIZE]
            ret = returns[i:i+BATCH_SIZE]

            log_probs, values, entropy = model.evaluate(s, a)

            ratio = torch.exp(log_probs - old_lp)
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (ret - values).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            writer.add_scalar("Loss/actor", actor_loss.item(), global_step)
            writer.add_scalar("Loss/critic", critic_loss.item(), global_step)
            writer.add_scalar("Loss/total", loss.item(), global_step)
            writer.add_scalar("Stats/entropy", entropy.mean().item(), global_step)


envs = [make_env(i+1,mode="still") for i in range(NUM_ENVS)]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n

model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
try:
    for episode in range(3000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_still_3000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()


In [11]:
envs = [make_env(i+1,mode="random") for i in range(NUM_ENVS)]
try:
    for episode in range(3000,9000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_random_6000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
    writer.close()

I0000 00:00:1776358839.727145  921634 fork_posix.cc:75] Other threads are currently calling into gRPC, skipping fork() handlers


[INFO] Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


INFO:mlagents_envs.environment:Connected to Unity environment with package version 2.1.0-exp.1 and communication version 1.5.0


[INFO] Connected new brain: SoccerTwos?team=1


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=1


[INFO] Connected new brain: SoccerTwos?team=0


INFO:mlagents_envs.environment:Connected new brain: SoccerTwos?team=0


Saved checkpoint at episode 3049
Saved checkpoint at episode 3099
Saved checkpoint at episode 3149
Saved checkpoint at episode 3199
Saved checkpoint at episode 3249
Saved checkpoint at episode 3299
Saved checkpoint at episode 3349
Saved checkpoint at episode 3399
Saved checkpoint at episode 3449
Saved checkpoint at episode 3499
Saved checkpoint at episode 3549
Saved checkpoint at episode 3599
Saved checkpoint at episode 3649
Saved checkpoint at episode 3699
Saved checkpoint at episode 3749
Saved checkpoint at episode 3799
Saved checkpoint at episode 3849
Saved checkpoint at episode 3899
Saved checkpoint at episode 3949
Saved checkpoint at episode 3999
Saved checkpoint at episode 4049
Saved checkpoint at episode 4099
Saved checkpoint at episode 4149
Saved checkpoint at episode 4199
Saved checkpoint at episode 4249
Saved checkpoint at episode 4299
Saved checkpoint at episode 4349
Saved checkpoint at episode 4399
Saved checkpoint at episode 4449
Saved checkpoint at episode 4499
Saved chec

In [ ]:

episode_rewards = []
current_rewards = [0 for _ in range(NUM_ENVS)]
global_step = 0


def collect_rollout(envs,model):
    buffer = RolloutBuffer()
    global global_step
    states = []
    for env in envs:
        states.append(env.reset()[:obs_dim])
    prev_ball_vels = [np.zeros(2) for _ in range(NUM_ENVS)]
    states = np.array(states)
    team_signs = [None for _ in range(NUM_ENVS)]

    for _ in range(ROLLOUT_STEPS):
        state_tensor = torch.tensor(states, dtype=torch.float32)

        with torch.no_grad():
            actions, log_probs, values = model.act(state_tensor)

        next_states = []
        rewards = []
        dones = []

        for i, env  in enumerate(envs):
            action = actions[i].item()
            obs, reward, done, info = env.step(action)
            if team_signs[i] is None:
                player_x = info["player_info"]["position"][0]
                team_signs[i] = +1 if player_x < 0 else -1
                
            ball_vel = info["ball_info"]["velocity"]
            r_pos, r_move, r_goal, r_dir_impact, r_impact = info_reward(
                info,
                prev_ball_vels[i],
                team_signs[i]
            )
            prev_ball_vels[i] = ball_vel.copy()
            r = reward + r_pos + r_move + r_goal + r_dir_impact + r_impact
            writer.add_scalar("reward/env", reward, global_step)
            writer.add_scalar("reward/move", r_move, global_step)
            writer.add_scalar("reward/pos", r_pos, global_step)
            writer.add_scalar("reward/goal", r_goal, global_step)
            writer.add_scalar("reward/impact_dir", r_dir_impact, global_step)
            writer.add_scalar("reward/impact", r_impact, global_step)
            writer.add_scalar("reward/total", r, global_step)
            if done:
                next_obs = env.reset()
                team_signs[i] = None

            else:
                next_obs = obs


            next_states.append(next_obs[:obs_dim])
            rewards.append(r)
            dones.append(done)

        buffer.states.extend(state_tensor)
        buffer.actions.extend(actions)
        buffer.log_probs.extend(log_probs)
        buffer.rewards.extend(rewards)
        buffer.dones.extend(dones)
        buffer.values.extend(values.detach().view(-1).tolist())

        states = np.array(next_states)
        global_step += NUM_ENVS

    return buffer


def ppo_update(buffer,optimizer):
    returns, advantages = buffer.compute_returns_advantages(GAMMA, LAMBDA)

    states = torch.stack(buffer.states)
    actions = torch.stack(buffer.actions)
    old_log_probs = torch.stack(buffer.log_probs)

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    for _ in range(EPOCHS):
        for i in range(0, len(states), BATCH_SIZE):
            s = states[i:i+BATCH_SIZE]
            a = actions[i:i+BATCH_SIZE]
            old_lp = old_log_probs[i:i+BATCH_SIZE]
            adv = advantages[i:i+BATCH_SIZE]
            ret = returns[i:i+BATCH_SIZE]

            log_probs, values, entropy = model.evaluate(s, a)

            ratio = torch.exp(log_probs - old_lp)
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv

            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = (ret - values).pow(2).mean()

            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            writer.add_scalar("Loss/actor", actor_loss.item(), global_step)
            writer.add_scalar("Loss/critic", critic_loss.item(), global_step)
            writer.add_scalar("Loss/total", loss.item(), global_step)
            writer.add_scalar("Stats/entropy", entropy.mean().item(), global_step)


envs = [make_env(i+1) for i in range(NUM_ENVS)]
flatteners = [ActionFlattener(env.action_space.nvec) for env in envs]
obs_dim = envs[0].observation_space.shape[0]
act_dim = envs[0].action_space.n

model = ActorCritic(obs_dim, act_dim)
optimizer = optim.Adam(model.parameters(), lr=LR)
try:
    for episode in range(3000):
        buffer = collect_rollout(envs,model)
        ppo_update(buffer,optimizer)

        episode_reward = sum(buffer.rewards)
        writer.add_scalar("episode/total_reward", episode_reward, episode)
        if episode % 50 == 49:
            torch.save(model.state_dict(), "ppo_checkpoint_still_3000.pth")
            print(f"Saved checkpoint at episode {episode}")
finally:
    for env in envs:
        env.close()
